In [14]:
# ╔══════════════════════════════════════════════════════════╗
# ║           Feature Engineering — Load Data               ║
# ╚══════════════════════════════════════════════════════════╝

import pandas as pd
import numpy as np

INPUT_PATH = "../data/processed/feature_selected.csv"

df = pd.read_csv(INPUT_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5600, 11)


,failed_connection_attempts,packets_sent_per_sec,packets_recv_per_sec,tcp_connection_count,context_switch_rate,unique_dst_port_count,top_process_cpu_pct,cpu_user_pct,load_avg_1m,attack_type,is_attack
0,0.0,0.0,0.0,9.0,567.0,0,0.0,6.21,0.18,crypto,1
1,0.0,0.0,0.0,9.0,803.0,0,0.0,1.50,0.41,crypto,1
2,0.0,0.0,0.0,10.0,893.0,0,0.0,5.68,0.41,crypto,1
3,0.0,0.0,0.0,14.0,404.0,0,26.0,4.99,0.41,crypto,1
4,0.0,0.0,0.0,14.0,496.0,0,59.9,2.17,0.37,crypto,1


In [15]:
# ╔══════════════════════════════════════════════════════════╗
# ║           Base Setup                                   ║
# ╚══════════════════════════════════════════════════════════╝

df_fe = df.copy()

# Ensure sorted data (important for rolling)
df_fe = df_fe.reset_index(drop=True)

print("Ready for feature engineering:", df_fe.shape)

Ready for feature engineering: (5600, 11)


In [16]:
# ╔══════════════════════════════════════════════════════════╗
# ║           Spike Detection Features                     ║
# ╚══════════════════════════════════════════════════════════╝

# Total packet activity
df_fe['packet_rate_total'] = (
    df_fe['packets_sent_per_sec'] +
    df_fe['packets_recv_per_sec']
)

# Packet spike ratio
df_fe['packet_spike_ratio'] = (
    df_fe['packet_rate_total'] /
    df_fe['packet_rate_total'].rolling(10, min_periods=1).mean()
)

# Connection spike
df_fe['connection_spike_ratio'] = (
    df_fe['tcp_connection_count'] /
    df_fe['tcp_connection_count'].rolling(10, min_periods=1).mean()
)

print("Spike features added")

Spike features added


In [17]:
# ╔══════════════════════════════════════════════════════════╗
# ║           CPU Behavior Features                        ║
# ╚══════════════════════════════════════════════════════════╝

# CPU spike
df_fe['cpu_spike_ratio'] = (
    df_fe['cpu_user_pct'] /
    df_fe['cpu_user_pct'].rolling(10, min_periods=1).mean()
)

# Process dominance
df_fe['process_cpu_ratio'] = (
    df_fe['top_process_cpu_pct'] /
    (df_fe['cpu_user_pct'] + 1e-5)
)

print("CPU features added")

CPU features added


In [18]:
# ╔══════════════════════════════════════════════════════════╗
# ║           Network Behavior Features                    ║
# ╚══════════════════════════════════════════════════════════╝

# Failure rate
df_fe['failure_rate'] = (
    df_fe['failed_connection_attempts'] /
    (df_fe['tcp_connection_count'] + 1)
)

# Port scan intensity
df_fe['port_scan_intensity'] = (
    df_fe['unique_dst_port_count'] /
    (df_fe['tcp_connection_count'] + 1)
)

# Load per connection
df_fe['load_per_connection'] = (
    df_fe['load_avg_1m'] /
    (df_fe['tcp_connection_count'] + 1)
)

print("Behavioral features added")

Behavioral features added


In [19]:
# ╔══════════════════════════════════════════════════════════╗
# ║           Final Engineered Dataset                     ║
# ╚══════════════════════════════════════════════════════════╝

print("\nFinal dataset shape:", df_fe.shape)

new_features = [col for col in df_fe.columns if col not in df.columns]

print("\nNew features created:")
print(new_features)

df_fe.head()


Final dataset shape: (5600, 19)

New features created:
['packet_rate_total', 'packet_spike_ratio', 'connection_spike_ratio', 'cpu_spike_ratio', 'process_cpu_ratio', 'failure_rate', 'port_scan_intensity', 'load_per_connection']


,failed_connection_attempts,packets_sent_per_sec,packets_recv_per_sec,tcp_connection_count,context_switch_rate,unique_dst_port_count,top_process_cpu_pct,cpu_user_pct,load_avg_1m,attack_type,is_attack,packet_rate_total,packet_spike_ratio,connection_spike_ratio,cpu_spike_ratio,process_cpu_ratio,failure_rate,port_scan_intensity,load_per_connection
0,0.0,0.0,0.0,9.0,567.0,0,0.0,6.21,0.18,crypto,1,0.0,NaN,1.000000,1.000000,0.000000,0.0,0.0,0.018000
1,0.0,0.0,0.0,9.0,803.0,0,0.0,1.50,0.41,crypto,1,0.0,NaN,1.000000,0.389105,0.000000,0.0,0.0,0.041000
2,0.0,0.0,0.0,10.0,893.0,0,0.0,5.68,0.41,crypto,1,0.0,NaN,1.071429,1.272591,0.000000,0.0,0.0,0.037273
3,0.0,0.0,0.0,14.0,404.0,0,26.0,4.99,0.41,crypto,1,0.0,NaN,1.333333,1.085963,5.210410,0.0,0.0,0.027333
4,0.0,0.0,0.0,14.0,496.0,0,59.9,2.17,0.37,crypto,1,0.0,NaN,1.250000,0.527981,27.603559,0.0,0.0,0.024667


In [20]:
import numpy as np
print("Infinity values:\n")
print(np.isinf(df_fe.select_dtypes(include=['number'])).sum())

Infinity values:

failed_connection_attempts    0
packets_sent_per_sec          0
packets_recv_per_sec          0
tcp_connection_count          0
context_switch_rate           0
unique_dst_port_count         0
top_process_cpu_pct           0
cpu_user_pct                  0
load_avg_1m                   0
is_attack                     0
packet_rate_total             0
packet_spike_ratio            0
connection_spike_ratio        0
cpu_spike_ratio               0
process_cpu_ratio             0
failure_rate                  0
port_scan_intensity           0
load_per_connection           0
dtype: int64


In [21]:
# ╔══════════════════════════════════════════════════════════╗
# ║           Handle Infinity and NaN Values                ║
# ╚══════════════════════════════════════════════════════════╝

# Replace infinity values with NaN first
df_fe = df_fe.replace([np.inf, -np.inf], np.nan)

# Fill NaN values with the median of each numeric column only
df_fe = df_fe.fillna(df_fe.median(numeric_only=True))

print("Infinity and NaN values cleaned")
print("\nRemaining NaN values:\n", df_fe.isnull().sum())
print("\nRemaining Infinity values:\n", np.isinf(df_fe.select_dtypes(include=['number'])).sum())

Infinity and NaN values cleaned

Remaining NaN values:
 failed_connection_attempts    0
packets_sent_per_sec          0
packets_recv_per_sec          0
tcp_connection_count          0
context_switch_rate           0
unique_dst_port_count         0
top_process_cpu_pct           0
cpu_user_pct                  0
load_avg_1m                   0
attack_type                   0
is_attack                     0
packet_rate_total             0
packet_spike_ratio            0
connection_spike_ratio        0
cpu_spike_ratio               0
process_cpu_ratio             0
failure_rate                  0
port_scan_intensity           0
load_per_connection           0
dtype: int64

Remaining Infinity values:
 failed_connection_attempts    0
packets_sent_per_sec          0
packets_recv_per_sec          0
tcp_connection_count          0
context_switch_rate           0
unique_dst_port_count         0
top_process_cpu_pct           0
cpu_user_pct                  0
load_avg_1m                   0
is_att

In [22]:
import numpy as np
print("Infinity values:\n")
print(np.isinf(df_fe.select_dtypes(include=['number'])).sum())

Infinity values:

failed_connection_attempts    0
packets_sent_per_sec          0
packets_recv_per_sec          0
tcp_connection_count          0
context_switch_rate           0
unique_dst_port_count         0
top_process_cpu_pct           0
cpu_user_pct                  0
load_avg_1m                   0
is_attack                     0
packet_rate_total             0
packet_spike_ratio            0
connection_spike_ratio        0
cpu_spike_ratio               0
process_cpu_ratio             0
failure_rate                  0
port_scan_intensity           0
load_per_connection           0
dtype: int64


In [23]:
SAVE_PATH = "../data/processed/engineered_dataset.csv"

df_fe.to_csv(SAVE_PATH, index=False)

print(f"\nSaved → {SAVE_PATH}")


Saved → ../data/processed/engineered_dataset.csv


In [24]:
df_fe.groupby('attack_type')[[
    'failure_rate',
    'port_scan_intensity',
    'process_cpu_ratio',
    'packet_spike_ratio'
]].mean()

,failure_rate,port_scan_intensity,process_cpu_ratio,packet_spike_ratio
attack_type,,,,
crypto,0.000000,0.000000,50986.835400,0.900669
ddos,0.115944,0.000000,0.001347,1.027723
none,0.000000,0.034059,0.023368,0.978410
portscan,0.000000,2.079050,0.004354,1.023718


In [25]:
print(df_fe.isnull().sum())

failed_connection_attempts    0
packets_sent_per_sec          0
packets_recv_per_sec          0
tcp_connection_count          0
context_switch_rate           0
unique_dst_port_count         0
top_process_cpu_pct           0
cpu_user_pct                  0
load_avg_1m                   0
attack_type                   0
is_attack                     0
packet_rate_total             0
packet_spike_ratio            0
connection_spike_ratio        0
cpu_spike_ratio               0
process_cpu_ratio             0
failure_rate                  0
port_scan_intensity           0
load_per_connection           0
dtype: int64
